# F2. Vulnerablities Identification

Once the ``cpe_whitelist.csv`` has been compiled, the CVE API can be called to ingest CVEs relating to the user-defined CPEs.

## Intended Purpose of Code

The below code was generated by AI and modified by the developer to create a simple to use tool that interacts with the NVD CVE API.

### Key features:

* Loads ``cpe_whitelist.csv`` generated in __F1. Asset Inventory__
    * Either loads to a dataframe or saves to a new file for ingested CVE data to be appended to
        * Regardless of loading to dataframe or saving to new file, ingested CVE data will need to be joined to corresponding CPE
            * Due to this, duplicate CVEs are to be expected and necessary for risk scoring granularity
* Ingests CVEs according to CPEs recorded in ``cpe_whitlelist.csv``
    * Columns for CVE data should reflect types of information discussed in the __Information returned__ section above
* Joins CVE data to corresponding CPEs
    * A surrogate key is generated for the joined dataset
    * A new row will be created for each CVE that corresponds to one (1) CPE

To handle cases where a CPE from the whitelist has no associated CVEs, a record must be generated that explicitly inserts a row for every CPE that returns no CVEs. This allows users to:
* Track all inventoried assets, even those with zero known vulnerabilities.
* Preserve one-to-one continuity between asset inventory (whitelist) and vulnerability table.
* Avoid accidental data loss or gaps in your final reports.

## Known Issues

* ``exploitabilityScore`` and ``impactScore`` do not return in the columns output when CVE–CPE rows are collected
* In the data processing cells to retrieve the missing ``exploitabilityScore`` and ``impactScore``:
    * the ``DataFrame`` generated in the API retrieval must first be saved and reloaded
        * for some reason the code is unsuccessful at pulling out these scores (all values return as 'None')
        * but does successfully return scores when the data is loaded from a csv file

In [1]:
import os, time, requests, pandas as pd
from pathlib import Path
import ast

# config
api_url      = "https://services.nvd.nist.gov/rest/json/cves/2.0"
api_key      = os.getenv("NVD_api_key")
whitelist    = Path("../data/cpe_whitelist.csv")
rate_secs    = 1.0
per_page     = 2000
progress_every = 25
#----------------------------------------------------------------------------------

def fetch_cves_for_cpe(cpe_uri: str) -> list[dict]:
    parts = cpe_uri.split(":")
    if len(parts) < 6:
        return []
    cpe_query = ":".join(parts[:6]) if parts[5] == "*" else cpe_uri
    all_items, start = [], 0
    headers = {"apiKey": api_key}
    while True:
        params = {
            "cpeName": cpe_query,
            "resultsPerPage": per_page,
            "startIndex": start,
        }
        r = requests.get(api_url, headers=headers, params=params, timeout=30)
        if r.status_code != 200:
            print(f"⚠️ {cpe_query[:70]} → {r.status_code}")
            break
        data = r.json()
        items = data.get("vulnerabilities", [])
        all_items.extend(items)
        start += per_page
        if start >= data.get("totalResults", 0) or not items:
            break
        time.sleep(rate_secs)
    return all_items

def flatten(v: dict, cpe_uri: str) -> dict:
    cve      = v["cve"]
    metrics  = cve.get("metrics", {})
    cvss31   = metrics.get("cvssMetricV31", [{}])[0].get("cvssData", {})
    cvss30   = metrics.get("cvssMetricV30", [{}])[0].get("cvssData", {})
    cvss     = cvss31 or cvss30
    descr    = next((d["value"] for d in cve.get("descriptions", []) if d["lang"] == "en"), "")
    cwes     = [
        d["value"] for w in cve.get("weaknesses", [])
        for d in w.get("description", []) if d.get("lang") == "en"
    ]
    refs     = " | ".join(r["url"] for r in cve.get("references", [])[:10])
    tags = ", ".join(tag for r in cve.get("references", [])[:10] for tag in r.get("tags", []))
    return {
        "cveID": cve["id"],
        "cpeName": cpe_uri,
        "published": cve.get("published"),
        "last_modified": cve.get("lastModified"),
        "vectorString": cvss.get("vectorString"),
        "baseScore": cvss.get("baseScore"),
        "baseSeverity": cvss.get("baseSeverity"),
        "attackVector": cvss.get("attackVector"),
        "attackComplexity": cvss.get("attackComplexity"),
        "privilegesRequired": cvss.get("privilegesRequired"),
        "userInteraction": cvss.get("userInteraction"),
        "scope": cvss.get("scope"),
        "confidentialityImpact": cvss.get("confidentialityImpact"),
        "integrityImpact": cvss.get("integrityImpact"),
        "availabilityImpact": cvss.get("availabilityImpact"),
        "cwes": ";".join(cwes) if cwes else None,
        "description": descr[:1000],
        "references": refs,
        "tags": tags,
        "full_json": v,
    }

# -- 1. load whitelist --------------------------------------------------
assets = pd.read_csv(whitelist, dtype=str)
cpe_list = assets["cpeName"].dropna().unique()
print(f"  {len(cpe_list):,} unique CPEs to query")

# -- 2. query API, record CPEs with no CVEs -----------------------------
rows = []
for idx, cpe in enumerate(cpe_list, start=1):
    if idx % progress_every == 0 or idx == 1:
        print(f"  → {idx}/{len(cpe_list)}   {cpe[:70]}…")
    vulns = fetch_cves_for_cpe(cpe)
    if vulns:
        for vuln in vulns:
            rows.append(flatten(vuln, cpe))
    else:
        # Insert row for CPEs with no CVEs
        rows.append({
            "cveID": None,
            "cpeName": cpe,
            "published": None,
            "last_modified": None,
            "vectorString": None,
            "baseScore": None,
            "baseSeverity": None,
            "attackVector": None,
            "attackComplexity": None,
            "privilegesRequired": None,
            "userInteraction": None,
            "scope": None,
            "confidentialityImpact": None,
            "integrityImpact": None,
            "availabilityImpact": None,
            "cwes": None,
            "description": "NO CVEs FOUND FOR THIS ASSET",
            "references": None,
            "tags": "NO CVEs",
            "full_json": None
        })

print("  API queries finished")

# -- 3. build DataFrame & de-dup --------------------------
df = (
    pd.DataFrame(rows)
      .drop_duplicates(subset=["cveID", "cpeName"])
      .reset_index(drop=True)
)
print(f"  {df.shape[0]:,} CVE–CPE rows collected")

  14 unique CPEs to query
  → 1/14   cpe:2.3:a:tableau:tableau_desktop:2021.1:*:*:*:*:*:*:*…
  API queries finished
  442 CVE–CPE rows collected


### Check that everything looks ok

In [2]:
df.head(4)

,cveID,cpeName,published,last_modified,vectorString,baseScore,baseSeverity,attackVector,attackComplexity,privilegesRequired,userInteraction,scope,confidentialityImpact,integrityImpact,availabilityImpact,cwes,description,references,tags,full_json
0,None,cpe:2.3:a:tableau:tableau_desktop:2021.1:*:*:*...,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,NO CVEs FOUND FOR THIS ASSET,None,NO CVEs,None
1,CVE-2021-39836,cpe:2.3:a:adobe:acrobat_reader:20.004.30006:*:...,2021-09-29T16:15:08.513,2024-11-21T06:20:20.730,CVSS:3.1/AV:L/AC:L/PR:N/UI:R/S:U/C:H/I:H/A:H,7.8,HIGH,LOCAL,LOW,NONE,REQUIRED,UNCHANGED,HIGH,HIGH,HIGH,CWE-416,Acrobat Reader DC versions 2021.005.20060 (and...,https://helpx.adobe.com/security/products/acro...,"Release Notes, Vendor Advisory, Release Notes,...","{'cve': {'id': 'CVE-2021-39836', 'sourceIdenti..."
2,CVE-2021-39837,cpe:2.3:a:adobe:acrobat_reader:20.004.30006:*:...,2021-09-29T16:15:08.573,2024-11-21T06:20:20.890,CVSS:3.0/AV:L/AC:L/PR:N/UI:R/S:U/C:H/I:H/A:H,7.8,HIGH,LOCAL,LOW,NONE,REQUIRED,UNCHANGED,HIGH,HIGH,HIGH,CWE-416,Acrobat Reader DC versions 2021.005.20060 (and...,https://helpx.adobe.com/security/products/acro...,"Release Notes, Vendor Advisory, Release Notes,...","{'cve': {'id': 'CVE-2021-39837', 'sourceIdenti..."
3,CVE-2021-39838,cpe:2.3:a:adobe:acrobat_reader:20.004.30006:*:...,2021-09-29T16:15:08.633,2024-11-21T06:20:21.040,CVSS:3.0/AV:L/AC:L/PR:N/UI:R/S:U/C:H/I:H/A:H,7.8,HIGH,LOCAL,LOW,NONE,REQUIRED,UNCHANGED,HIGH,HIGH,HIGH,CWE-416,Acrobat Reader DC versions 2021.005.20060 (and...,https://helpx.adobe.com/security/products/acro...,"Release Notes, Vendor Advisory, Release Notes,...","{'cve': {'id': 'CVE-2021-39838', 'sourceIdenti..."


In [3]:
# demonstrates how a no-cve-returned scenario is handled
df[df['description'] == "NO CVEs FOUND FOR THIS ASSET"]

,cveID,cpeName,published,last_modified,vectorString,baseScore,baseSeverity,attackVector,attackComplexity,privilegesRequired,userInteraction,scope,confidentialityImpact,integrityImpact,availabilityImpact,cwes,description,references,tags,full_json
0,None,cpe:2.3:a:tableau:tableau_desktop:2021.1:*:*:*...,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,NO CVEs FOUND FOR THIS ASSET,None,NO CVEs,None
318,None,cpe:2.3:a:oracle:suitecommerce_advanced:2020.1...,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,NO CVEs FOUND FOR THIS ASSET,None,NO CVEs,None


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   cveID                  440 non-null    object 
 1   cpeName                442 non-null    object 
 2   published              440 non-null    object 
 3   last_modified          440 non-null    object 
 4   vectorString           421 non-null    object 
 5   baseScore              421 non-null    float64
 6   baseSeverity           421 non-null    object 
 7   attackVector           421 non-null    object 
 8   attackComplexity       421 non-null    object 
 9   privilegesRequired     421 non-null    object 
 10  userInteraction        421 non-null    object 
 11  scope                  421 non-null    object 
 12  confidentialityImpact  421 non-null    object 
 13  integrityImpact        421 non-null    object 
 14  availabilityImpact     421 non-null    object 
 15  cwes  

In [5]:
df.describe(include='all')

,cveID,cpeName,published,last_modified,vectorString,baseScore,baseSeverity,attackVector,attackComplexity,privilegesRequired,userInteraction,scope,confidentialityImpact,integrityImpact,availabilityImpact,cwes,description,references,tags,full_json
count,440,442,440,440,421,421.000000,421,421,421,421,421,421,421,421,421,440,442,440,442,440
unique,436,14,435,424,82,NaN,4,4,2,3,2,2,3,3,3,64,258,117,45,436
top,CVE-1999-1322,cpe:2.3:a:adobe:acrobat_reader:20.004.30006:*:...,2012-10-18T17:55:01.613,2025-04-03T01:03:51.193,CVSS:3.1/AV:L/AC:L/PR:N/UI:R/S:U/C:H/I:H/A:H,NaN,HIGH,LOCAL,LOW,NONE,REQUIRED,UNCHANGED,HIGH,HIGH,HIGH,CWE-416,Acrobat Reader DC version 22.001.2011x (and ea...,https://helpx.adobe.com/security/products/acro...,"Vendor Advisory, Vendor Advisory","{'cve': {'id': 'CVE-1999-1322', 'sourceIdentif..."
freq,2,315,2,8,150,NaN,207,322,390,352,351,394,326,225,235,97,20,65,181,2
mean,NaN,NaN,NaN,NaN,NaN,6.558907,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,1.525232,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,2.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,5.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,7.200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,7.800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Missing exploitabilityScore and impactScore

These two scores are very likely important for developing the risk scoring model, so these will need to be pulled out of the ``full_json`` column somehow.

In [6]:
# for some reason the code in the cell below is unsuccessful at pulling out these scores (all values return as 'None')
# but does successfully return scores when the data is loaded from a csv file
df.to_csv('../data/vuln_catalogue_v1.csv')

In [7]:
import pandas as pd
import ast

df = pd.read_csv('../data/vuln_catalogue_v1.csv')

def parse_mixed_json(s):
    try:
        return ast.literal_eval(s)
    except Exception:
        return {}

def extract_scores(full_json):
    data = parse_mixed_json(full_json)
    for key in ['cvssMetricV31', 'cvssMetricV30']:
        try:
            entry = data['cve']['metrics'][key][0]
            exploit = entry.get('exploitabilityScore', None)
            impact = entry.get('impactScore', None)
            if exploit is not None or impact is not None:
                return pd.Series({'exploitabilityScore': exploit, 'impactScore': impact})
        except Exception:
            continue
    # fallback: not found
    return pd.Series({'exploitabilityScore': None, 'impactScore': None})

df[['exploitabilityScore', 'impactScore']] = df['full_json'].apply(extract_scores)

# Optional: Save or inspect
#df.to_csv('vuln_catalogue_with_scores.csv', index=False)
df[['exploitabilityScore', 'impactScore']]

,exploitabilityScore,impactScore
0,NaN,NaN
1,1.8,5.9
2,1.8,5.9
3,1.8,5.9
4,1.8,5.9
...,...,...
437,NaN,NaN
438,NaN,NaN
439,NaN,NaN
440,NaN,NaN


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             442 non-null    int64  
 1   cveID                  440 non-null    object 
 2   cpeName                442 non-null    object 
 3   published              440 non-null    object 
 4   last_modified          440 non-null    object 
 5   vectorString           421 non-null    object 
 6   baseScore              421 non-null    float64
 7   baseSeverity           421 non-null    object 
 8   attackVector           421 non-null    object 
 9   attackComplexity       421 non-null    object 
 10  privilegesRequired     421 non-null    object 
 11  userInteraction        421 non-null    object 
 12  scope                  421 non-null    object 
 13  confidentialityImpact  421 non-null    object 
 14  integrityImpact        421 non-null    object 
 15  availa

In [9]:
# Assume df is your DataFrame and columns already exist

cols = df.columns.tolist()
# Get the index of 'baseScore'
base_idx = cols.index('baseScore')

# Remove if they're already present elsewhere
cols.remove('exploitabilityScore')
cols.remove('impactScore')

# Insert them in the right spot
cols = cols[:base_idx+1] + ['exploitabilityScore', 'impactScore'] + cols[base_idx+1:]

# Reorder the DataFrame
df = df[cols]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             442 non-null    int64  
 1   cveID                  440 non-null    object 
 2   cpeName                442 non-null    object 
 3   published              440 non-null    object 
 4   last_modified          440 non-null    object 
 5   vectorString           421 non-null    object 
 6   baseScore              421 non-null    float64
 7   exploitabilityScore    421 non-null    float64
 8   impactScore            421 non-null    float64
 9   baseSeverity           421 non-null    object 
 10  attackVector           421 non-null    object 
 11  attackComplexity       421 non-null    object 
 12  privilegesRequired     421 non-null    object 
 13  userInteraction        421 non-null    object 
 14  scope                  421 non-null    object 
 15  confid

In [10]:
df.drop(columns='Unnamed: 0', inplace=True, axis=1)
df

,cveID,cpeName,published,last_modified,vectorString,baseScore,exploitabilityScore,impactScore,baseSeverity,attackVector,...,userInteraction,scope,confidentialityImpact,integrityImpact,availabilityImpact,cwes,description,references,tags,full_json
0,NaN,cpe:2.3:a:tableau:tableau_desktop:2021.1:*:*:*...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NO CVEs FOUND FOR THIS ASSET,NaN,NO CVEs,NaN
1,CVE-2021-39836,cpe:2.3:a:adobe:acrobat_reader:20.004.30006:*:...,2021-09-29T16:15:08.513,2024-11-21T06:20:20.730,CVSS:3.1/AV:L/AC:L/PR:N/UI:R/S:U/C:H/I:H/A:H,7.8,1.8,5.9,HIGH,LOCAL,...,REQUIRED,UNCHANGED,HIGH,HIGH,HIGH,CWE-416,Acrobat Reader DC versions 2021.005.20060 (and...,https://helpx.adobe.com/security/products/acro...,"Release Notes, Vendor Advisory, Release Notes,...","{'cve': {'id': 'CVE-2021-39836', 'sourceIdenti..."
2,CVE-2021-39837,cpe:2.3:a:adobe:acrobat_reader:20.004.30006:*:...,2021-09-29T16:15:08.573,2024-11-21T06:20:20.890,CVSS:3.0/AV:L/AC:L/PR:N/UI:R/S:U/C:H/I:H/A:H,7.8,1.8,5.9,HIGH,LOCAL,...,REQUIRED,UNCHANGED,HIGH,HIGH,HIGH,CWE-416,Acrobat Reader DC versions 2021.005.20060 (and...,https://helpx.adobe.com/security/products/acro...,"Release Notes, Vendor Advisory, Release Notes,...","{'cve': {'id': 'CVE-2021-39837', 'sourceIdenti..."
3,CVE-2021-39838,cpe:2.3:a:adobe:acrobat_reader:20.004.30006:*:...,2021-09-29T16:15:08.633,2024-11-21T06:20:21.040,CVSS:3.0/AV:L/AC:L/PR:N/UI:R/S:U/C:H/I:H/A:H,7.8,1.8,5.9,HIGH,LOCAL,...,REQUIRED,UNCHANGED,HIGH,HIGH,HIGH,CWE-416,Acrobat Reader DC versions 2021.005.20060 (and...,https://helpx.adobe.com/security/products/acro...,"Release Notes, Vendor Advisory, Release Notes,...","{'cve': {'id': 'CVE-2021-39838', 'sourceIdenti..."
4,CVE-2021-39839,cpe:2.3:a:adobe:acrobat_reader:20.004.30006:*:...,2021-09-29T16:15:08.693,2024-11-21T06:20:21.190,CVSS:3.0/AV:L/AC:L/PR:N/UI:R/S:U/C:H/I:H/A:H,7.8,1.8,5.9,HIGH,LOCAL,...,REQUIRED,UNCHANGED,HIGH,HIGH,HIGH,CWE-416,Acrobat Reader DC versions 2021.005.20060 (and...,https://helpx.adobe.com/security/products/acro...,"Release Notes, Vendor Advisory, Release Notes,...","{'cve': {'id': 'CVE-2021-39839', 'sourceIdenti..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
437,CVE-1999-1322,cpe:2.3:a:microsoft:exchange_server:2019:cumul...,1998-11-12T05:00:00.000,2025-04-03T01:03:51.193,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NVD-CWE-Other,The installation of 1ArcServe Backup and Inocu...,http://marc.info/?l=ntbugtraq&m=91096758513985...,NaN,"{'cve': {'id': 'CVE-1999-1322', 'sourceIdentif..."
438,CVE-2000-0216,cpe:2.3:a:microsoft:exchange_server:2019:cumul...,2000-02-29T05:00:00.000,2025-04-03T01:03:51.193,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NVD-CWE-Other,"Microsoft email clients in Outlook, Exchange, ...",http://archives.neohapsis.com/archives/ntbugtr...,"Vendor Advisory, Vendor Advisory","{'cve': {'id': 'CVE-2000-0216', 'sourceIdentif..."
439,CVE-2011-0290,cpe:2.3:a:microsoft:exchange_server:2019:cumul...,2011-10-21T10:55:03.757,2025-04-11T00:51:21.963,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,CWE-264,The BlackBerry Collaboration Service in Resear...,http://secunia.com/advisories/46370 | http://s...,"Vendor Advisory, Exploit, Vendor Advisory, Ven...","{'cve': {'id': 'CVE-2011-0290', 'sourceIdentif..."
440,CVE-2012-2284,cpe:2.3:a:microsoft:exchange_server:2019:cumul...,2012-10-18T17:55:01.613,2025-04-11T00:51:21.963,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,CWE-255,The (1) install and (2) upgrade processes in E...,http://archives.neohapsis.com/archives/bugtraq...,NaN,"{'cve': {'id': 'CVE-2012-2284', 'sourceIdentif..."


In [11]:
assets

,WrittenAt,Title,cpeName
0,2025-06-08T18:58:54.485,Tableau Desktop 2021.1,cpe:2.3:a:tableau:tableau_desktop:2021.1:*:*:*...
1,2025-06-08T18:59:05.725,Adobe Acrobat Reader 20.004.30006 Classic Edition,cpe:2.3:a:adobe:acrobat_reader:20.004.30006:*:...
2,2025-06-08T18:59:17.606,Oracle SuiteCommerce Advanced,cpe:2.3:a:oracle:suitecommerce_advanced:-:*:*:...
3,2025-06-08T18:59:17.606,Oracle SuiteCommerce Advanced 2020.1.4,cpe:2.3:a:oracle:suitecommerce_advanced:2020.1...
4,2025-06-08T18:59:27.454,Alteryx Server 2022.1.1.42590,cpe:2.3:a:alteryx:alteryx_server:2022.1.1.4259...
5,2025-06-08T21:55:41.178,Fortinet FortiGate 7000,cpe:2.3:h:fortinet:fortigate_7000:-:*:*:*:*:*:*:*
6,2025-06-08T22:01:52.494,Workday 31.2,cpe:2.3:a:workday:workday:31.2:*:*:*:*:*:*:*
7,2025-06-08T22:04:13.394,Alfresco Enterprise 4.1.6.13,cpe:2.3:a:alfresco:alfresco:4.1.6.13:*:*:*:ent...
8,2025-06-08T22:06:35.432,Oracle Database 19c Enterprise Edition,cpe:2.3:a:oracle:database:19c:*:*:*:enterprise...
9,2025-06-08T22:06:53.654,Oracle Database Vault 19c,cpe:2.3:a:oracle:database_vault:19c:*:*:*:*:*:*:*


In [12]:
# merge cve outputs and assets to quickly know which asset corresponds to which cve
vuln = pd.merge(assets,df, how='inner', on='cpeName')
# add a surrogate key to respect relational database rules
vuln['sid'] = vuln.index
# move "sid" to the first column
vuln.insert(0, "sid", vuln.pop("sid"))
vuln.head(4)

,sid,WrittenAt,Title,cpeName,cveID,published,last_modified,vectorString,baseScore,exploitabilityScore,...,userInteraction,scope,confidentialityImpact,integrityImpact,availabilityImpact,cwes,description,references,tags,full_json
0,0,2025-06-08T18:58:54.485,Tableau Desktop 2021.1,cpe:2.3:a:tableau:tableau_desktop:2021.1:*:*:*...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NO CVEs FOUND FOR THIS ASSET,NaN,NO CVEs,NaN
1,1,2025-06-08T18:59:05.725,Adobe Acrobat Reader 20.004.30006 Classic Edition,cpe:2.3:a:adobe:acrobat_reader:20.004.30006:*:...,CVE-2021-39836,2021-09-29T16:15:08.513,2024-11-21T06:20:20.730,CVSS:3.1/AV:L/AC:L/PR:N/UI:R/S:U/C:H/I:H/A:H,7.8,1.8,...,REQUIRED,UNCHANGED,HIGH,HIGH,HIGH,CWE-416,Acrobat Reader DC versions 2021.005.20060 (and...,https://helpx.adobe.com/security/products/acro...,"Release Notes, Vendor Advisory, Release Notes,...","{'cve': {'id': 'CVE-2021-39836', 'sourceIdenti..."
2,2,2025-06-08T18:59:05.725,Adobe Acrobat Reader 20.004.30006 Classic Edition,cpe:2.3:a:adobe:acrobat_reader:20.004.30006:*:...,CVE-2021-39837,2021-09-29T16:15:08.573,2024-11-21T06:20:20.890,CVSS:3.0/AV:L/AC:L/PR:N/UI:R/S:U/C:H/I:H/A:H,7.8,1.8,...,REQUIRED,UNCHANGED,HIGH,HIGH,HIGH,CWE-416,Acrobat Reader DC versions 2021.005.20060 (and...,https://helpx.adobe.com/security/products/acro...,"Release Notes, Vendor Advisory, Release Notes,...","{'cve': {'id': 'CVE-2021-39837', 'sourceIdenti..."
3,3,2025-06-08T18:59:05.725,Adobe Acrobat Reader 20.004.30006 Classic Edition,cpe:2.3:a:adobe:acrobat_reader:20.004.30006:*:...,CVE-2021-39838,2021-09-29T16:15:08.633,2024-11-21T06:20:21.040,CVSS:3.0/AV:L/AC:L/PR:N/UI:R/S:U/C:H/I:H/A:H,7.8,1.8,...,REQUIRED,UNCHANGED,HIGH,HIGH,HIGH,CWE-416,Acrobat Reader DC versions 2021.005.20060 (and...,https://helpx.adobe.com/security/products/acro...,"Release Notes, Vendor Advisory, Release Notes,...","{'cve': {'id': 'CVE-2021-39838', 'sourceIdenti..."


In [13]:
vuln.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   sid                    442 non-null    int64  
 1   WrittenAt              442 non-null    object 
 2   Title                  442 non-null    object 
 3   cpeName                442 non-null    object 
 4   cveID                  440 non-null    object 
 5   published              440 non-null    object 
 6   last_modified          440 non-null    object 
 7   vectorString           421 non-null    object 
 8   baseScore              421 non-null    float64
 9   exploitabilityScore    421 non-null    float64
 10  impactScore            421 non-null    float64
 11  baseSeverity           421 non-null    object 
 12  attackVector           421 non-null    object 
 13  attackComplexity       421 non-null    object 
 14  privilegesRequired     421 non-null    object 
 15  userIn

In [14]:
# save to file
vuln.to_csv('../data/vuln_catalogue_v2.csv')